# 03 — Tokenization (MentalBERT)

**Project:** MentalBERT-CSSR  
**Goal:** Measure true subword sequence lengths with `mental/mental-bert-base-uncased`, set `truncation_side="left"`, and recommend `max_length` for training.

---

## Design decisions

| Decision | Choice | Why |
|----------|--------|-----|
| Tokenizer / encoder | **`mental/mental-bert-base-uncased` only** | Domain BERT for mental-health text; project lock |
| Loader | `BertTokenizer` (WordPiece) | Avoids Hub `chat_templates` 401s on newer `transformers`; still the same MentalBERT vocab |
| Input data | Processed CSV from Notebook 2 | Clean Unicode / whitespace without destroying clinical signal |
| Special tokens | Included in length stats | Training encode uses `[CLS]` / `[SEP]` |
| `truncation_side` | **`left`** | Crisis posts often put context first and **intent / plan / climax late**; left truncation drops the prefix and **keeps the ending** |
| `max_length` policy | Smallest candidate with ≥99% coverage (else ≥ p95) | Balance information retention vs. compute / padding |
| Hard ceiling | 512 | BERT absolute position limit |

### Why left truncation (clinical rationale)

Reddit r/SuicideWatch posts frequently follow a narrative arc:

1. Background / history (beginning)
2. Escalation
3. **Current intent, method ideation, or plea** (end)

Right truncation (Hugging Face default) keeps the beginning and cuts the end — exactly the span most aligned with C-SSRS severity.  
**Left truncation reverses this:** when a post exceeds `max_length`, the model still sees the clinically denser suffix.

> Encoder substitution (DistilBERT / RoBERTa / DeBERTa) is **forbidden** in this project line.

**Stop after this notebook** until Notebook 4 is approved.

## 1. Environment, configuration, and reproducibility

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent]
PROJECT_ROOT = None
for candidate in CANDIDATES:
    if (candidate / "configs" / "default.yaml").exists() and (candidate / "utils").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import ensure_directories, get_logger, load_config, set_seed
from utils.io import load_processed_dataset
from utils.tokenization import (
    MENTALBERT_MODEL_NAME,
    build_token_length_report,
    compute_token_lengths,
    demonstrate_left_truncation,
    load_mentalbert_tokenizer,
    plot_coverage_curve,
    plot_token_length_boxplot_by_class,
    plot_token_length_histogram,
    token_lengths_by_class,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 140)

cfg = load_config()
set_seed(cfg.SEED)
ensure_directories(cfg)
logger = get_logger("notebook.03_tokenization")

TEXT_COL = cfg.data.text_column
LABEL_COL = cfg.data.label_column
TOK = cfg.tokenization
PLOTS_DIR = Path(cfg.paths.plots_path)
METRICS_DIR = Path(cfg.paths.metrics_path)
DPI = int(cfg.eda.figure_dpi)

assert TOK.model_name == MENTALBERT_MODEL_NAME, "Model lock violated"
assert TOK.truncation_side == "left", "This project requires left truncation"

logger.info("Model          : %s", TOK.model_name)
logger.info("Truncation side: %s", TOK.truncation_side)
logger.info("Processed data : %s", cfg.paths.processed_data)

## 2. Load processed corpus

In [ ]:
df = load_processed_dataset(
    cfg.paths.processed_data,
    text_column=TEXT_COL,
    label_column=LABEL_COL,
)
print(f"Processed shape: {df.shape}")
display(df[[TEXT_COL, LABEL_COL]].head(3))
assert df[TEXT_COL].isna().sum() == 0
assert df[LABEL_COL].isna().sum() == 0

## 3. Load MentalBERT tokenizer (`truncation_side="left"`)

In [ ]:
tokenizer = load_mentalbert_tokenizer(
    model_name=TOK.model_name,
    truncation_side=TOK.truncation_side,
)

print("tokenizer class     :", tokenizer.__class__.__name__)
print("vocab size          :", tokenizer.vocab_size)
print("model_max_length    :", tokenizer.model_max_length)
print("truncation_side     :", tokenizer.truncation_side)
print("special tokens      :", {
    "cls": tokenizer.cls_token,
    "sep": tokenizer.sep_token,
    "pad": tokenizer.pad_token,
    "unk": tokenizer.unk_token,
})

assert tokenizer.truncation_side == "left"
print("Left truncation confirmed.")

## 4. Compute MentalBERT sequence lengths

Lengths include special tokens and are measured **without** truncation so we see the true distribution.

In [ ]:
texts = df[TEXT_COL].astype(str).tolist()
lengths = compute_token_lengths(
    texts,
    tokenizer,
    add_special_tokens=bool(TOK.add_special_tokens),
    batch_size=int(TOK.batch_size),
)

df_tok = df.copy()
df_tok["token_length"] = lengths

print(f"n_texts          : {len(lengths)}")
print(f"min / mean / max : {lengths.min()} / {lengths.mean():.2f} / {lengths.max()}")
print(f"median           : {np.median(lengths):.2f}")
print(f"p95              : {np.quantile(lengths, 0.95):.2f}")
print(f"p99              : {np.quantile(lengths, 0.99):.2f}")

display(df_tok[[TEXT_COL, LABEL_COL, "token_length"]].head(5))

## 5. Recommend `max_length`

In [ ]:
report = build_token_length_report(
    lengths,
    model_name=TOK.model_name,
    truncation_side=TOK.truncation_side,
    percentiles=list(cfg.eda.length_percentiles),
    candidates=list(TOK.candidate_max_lengths),
    target_coverage=float(TOK.target_coverage),
)
report_dict = report.to_dict()

print("TOKEN LENGTH SUMMARY")
display(pd.DataFrame([{
    "n_texts": report.n_texts,
    "min": report.min_tokens,
    "mean": round(report.mean_tokens, 3),
    "std": round(report.std_tokens, 3),
    "median": report.median_tokens,
    "max": report.max_tokens,
    **{k: round(v, 3) for k, v in report.percentiles.items()},
}]))

coverage_df = pd.DataFrame([
    {"max_length": int(k), "coverage_pct": round(v * 100, 4)}
    for k, v in sorted(report.coverage_by_candidate.items(), key=lambda x: int(x[0]))
])
print("COVERAGE BY CANDIDATE max_length")
display(coverage_df)

print("=" * 72)
print(f"RECOMMENDED max_length = {report.recommended_max_length}")
print(report.recommendation_rationale)
print(
    f"Sequences exceeding recommendation: "
    f"{report.n_exceeding_recommended} ({report.pct_exceeding_recommended:.3f}%)"
)
print("=" * 72)

# Provisional config value vs recommendation
print(f"Config provisional max_length : {cfg.MAX_LENGTH}")
print(f"Notebook recommendation       : {report.recommended_max_length}")
if int(cfg.MAX_LENGTH) != int(report.recommended_max_length):
    print(
        "NOTE: Update configs/default.yaml model.max_length to the recommendation "
        "before Notebook 4 (or load RESULTS/metrics/max_length_recommendation.json)."
    )

## 6. Lengths by severity class

In [ ]:
by_class = token_lengths_by_class(df_tok, lengths, label_column=LABEL_COL)
display(by_class.round(3))
by_class.to_csv(METRICS_DIR / "tokenization_lengths_by_severity.csv", index=False)

## 7. Visualisations

In [ ]:
plot_token_length_histogram(
    lengths,
    PLOTS_DIR / "tokenization_token_length_histogram.png",
    recommended_max_length=report.recommended_max_length,
    dpi=DPI,
)
plot_token_length_boxplot_by_class(
    df_tok,
    lengths,
    LABEL_COL,
    PLOTS_DIR / "tokenization_token_length_boxplot_by_severity.png",
    dpi=DPI,
)
plot_coverage_curve(
    report.coverage_by_candidate,
    PLOTS_DIR / "tokenization_coverage_vs_max_length.png",
    recommended_max_length=report.recommended_max_length,
    dpi=DPI,
)
print("Plots written to", PLOTS_DIR)

## 8. Left vs right truncation demo

Pick a long post (if any exceed the recommendation) and show that **left** truncation preserves the ending.

In [ ]:
demo_max = min(64, int(report.recommended_max_length))  # aggressive demo window
long_idx = int(np.argmax(lengths))
demo_text = str(df_tok.iloc[long_idx][TEXT_COL])
demo = demonstrate_left_truncation(demo_text, tokenizer, max_length=demo_max)

print(f"Full token length : {demo['full_length']}")
print(f"Demo max_length   : {demo['max_length']} (truncated={demo['truncated']})")
print(f"Left is suffix?   : {demo['left_decoded_is_suffix_of_full']}")
print(f"Right is prefix?  : {demo['right_decoded_is_prefix_of_full']}")
print()
print("--- LEFT truncation (keeps END) ---")
print(demo["left_truncation_decoded"][:500])
print()
print("--- RIGHT truncation (keeps START) ---")
print(demo["right_truncation_decoded"][:500])

# Restore project policy after the demo mutates temporarily inside the helper
tokenizer.truncation_side = "left"
assert tokenizer.truncation_side == "left"

pd.DataFrame([{
    "full_length": demo["full_length"],
    "demo_max_length": demo["max_length"],
    "left_is_suffix": demo["left_decoded_is_suffix_of_full"],
    "right_is_prefix": demo["right_decoded_is_prefix_of_full"],
    "left_decoded": demo["left_truncation_decoded"],
    "right_decoded": demo["right_truncation_decoded"],
}]).to_csv(METRICS_DIR / "tokenization_left_vs_right_demo.csv", index=False)

## 9. Persist recommendation, metrics, experiment log

Notebook 4 should read `RESULTS/metrics/max_length_recommendation.json` (or the updated YAML) so training uses the empirically chosen length.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Per-row lengths for audit / future error analysis
df_tok[[TEXT_COL, LABEL_COL, "token_length"]].to_csv(
    METRICS_DIR / "tokenization_per_example_lengths.csv", index=False
)
coverage_df.to_csv(METRICS_DIR / "tokenization_coverage_by_max_length.csv", index=False)

recommendation = {
    "timestamp_utc": timestamp,
    "stage": "tokenization",
    "notebook": "03_Tokenization.ipynb",
    "dataset_name": cfg.experiment.dataset_name,
    "model_name": TOK.model_name,
    "truncation_side": TOK.truncation_side,
    "recommended_max_length": int(report.recommended_max_length),
    "provisional_config_max_length": int(cfg.MAX_LENGTH),
    "rationale": report.recommendation_rationale,
    "token_length_report": report_dict,
    "hyperparameters_snapshot": {
        "learning_rate": cfg.LEARNING_RATE,
        "batch_size": cfg.BATCH_SIZE,
        "epochs": cfg.EPOCHS,
        "dropout": cfg.DROPOUT,
        "seed": cfg.SEED,
        "optimizer": cfg.training.optimizer,
        "scheduler": cfg.training.scheduler,
        "label_smoothing": cfg.LABEL_SMOOTHING,
        "warmup_ratio": cfg.WARMUP_RATIO,
    },
}

rec_path = Path(PROJECT_ROOT) / TOK.recommendation_path
rec_path.parent.mkdir(parents=True, exist_ok=True)
with rec_path.open("w", encoding="utf-8") as fh:
    json.dump(recommendation, fh, indent=2, ensure_ascii=False)

report_path = METRICS_DIR / "tokenization_report.json"
with report_path.open("w", encoding="utf-8") as fh:
    json.dump(recommendation, fh, indent=2, ensure_ascii=False)

# Optionally sync configs/default.yaml model.max_length to the recommendation
if bool(TOK.update_config_note) and int(cfg.MAX_LENGTH) != int(report.recommended_max_length):
    yaml_path = PROJECT_ROOT / "configs" / "default.yaml"
    text = yaml_path.read_text(encoding="utf-8")
    old = f"max_length: {int(cfg.MAX_LENGTH)}"
    new = f"max_length: {int(report.recommended_max_length)}          # set by Notebook 3 recommendation"
    if old in text:
        yaml_path.write_text(text.replace(old, new, 1), encoding="utf-8")
        print(f"Updated {yaml_path} → model.max_length={report.recommended_max_length}")
    else:
        print("Could not auto-patch YAML (pattern not found); set max_length manually.")

log_path = Path(cfg.paths.experiment_log)
with log_path.open("a", encoding="utf-8") as fh:
    fh.write(json.dumps({
        "timestamp_utc": timestamp,
        "stage": "tokenization",
        "notebook": "03_Tokenization.ipynb",
        "dataset_name": cfg.experiment.dataset_name,
        "model_name": TOK.model_name,
        "truncation_side": TOK.truncation_side,
        "recommended_max_length": int(report.recommended_max_length),
        "p95": report.percentiles.get("p95"),
        "p99": report.percentiles.get("p99"),
        "seed": cfg.SEED,
        "learning_rate": cfg.LEARNING_RATE,
        "batch_size": cfg.BATCH_SIZE,
        "epochs": cfg.EPOCHS,
        "dropout": cfg.DROPOUT,
        "optimizer": cfg.training.optimizer,
        "scheduler": cfg.training.scheduler,
    }, ensure_ascii=False) + "\n")

print("Saved:")
print(f"  {rec_path}")
print(f"  {report_path}")
print(f"  {log_path}")

## 10. Summary → Notebook 4

| Setting | Value |
|---------|-------|
| Tokenizer | `mental/mental-bert-base-uncased` |
| `truncation_side` | `left` |
| Recommended `max_length` | _see cell output / `max_length_recommendation.json`_ |

Training notebook must:

1. Load this tokenizer + recommendation
2. Encode with `truncation=True`, `padding="max_length"`, `truncation_side="left"`
3. Fine-tune MentalBERT only (no backbone swap)

---

**Stop here.** Await approval before generating Notebook 4 (Training).